In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.blocking import build_index, prepare_pool

D = "../dataset/dev/"

# WHY only dev files: the full raw tables would waste RAM and are not needed here
s1_dev = pd.read_csv(D + "s1_dev.tsv", sep="\t")
gt_dev = pd.read_csv(D + "gt_dev.tsv", sep="\t")
pool_df = pd.concat([pd.read_csv(D + "s2_dev.tsv", sep="\t"),
                     pd.read_csv(D + "s3_dev.tsv", sep="\t")], ignore_index=True)

# WHY header=None: these id files have no header row
val_ids = set(pd.read_csv(D + "val_ids.txt", header=None)[0])

# WHY the saved pairs: they are exactly what the matcher saw, so "offered" means "in these pairs"
val_pairs = pd.read_pickle(D + "val_pairs.pkl")

# WHY truth as sets: a singleton has no matches, so its set is empty
truth = {s: (set(m.split(",")) if isinstance(m, str) else set())
         for s, m in zip(gt_dev["source1_entity_id"], gt_dev["matched_entity_ids"])}

# WHY build once: indexing and cleaning the pool is the slow part
index = build_index(pool_df)
prepared = prepare_pool(pool_df)

print(len(s1_dev), len(pool_df), len(val_ids), len(val_pairs))
print("keys:", len(index))

100000 467494 20000 577150
keys: 1155909


In [2]:
# WHY a set of pairs: fast lookup to ask "did the matcher ever see this pair?"
offered = set(zip(val_pairs["s1"], val_pairs["cand"]))

# WHY only val entities: blocking is scored on these, never on the 80k training ones
misses = []   # (s1_id, missed_record_id)
for eid in val_ids:
    for t in truth[eid]:
        if (eid, t) not in offered:
            misses.append((eid, t))

total_true = sum(len(truth[e]) for e in val_ids)
print("true matches:", total_true)       # expect 69,455
print("misses:", len(misses))            # expect 7,166
print("share:", round(len(misses) / total_true, 4))   # expect about 0.1032

true matches: 69455
misses: 7166
share: 0.1032


In [3]:
from collections import defaultdict
from src.blocking import get_candidates_topk

# WHY group by S1: the first stage runs once per entity, not once per miss (faster)
by_s1 = defaultdict(list)
for eid, t in misses:
    by_s1[eid].append(t)

# WHY set_index: lets us read an S1 entity's name, address and country by its ID
s1_idx = s1_dev.set_index("entity_id")

cut_by_rerank = []     # in the top-100, lost at rerank
not_in_top100 = []     # blocking never ranked it in the top 100

for eid, lost in by_s1.items():
    row = s1_idx.loc[eid]
    # WHY same settings as build_pairs: cap 1000, top 100, or we would measure a different process
    first = get_candidates_topk(row["business_name"], row["business_address"],
                                row["country"], index, 1000, 100)
    for t in lost:
        (cut_by_rerank if t in first else not_in_top100).append((eid, t))

n = len(misses)
print("cut by rerank (was in top-100):", len(cut_by_rerank), f"({len(cut_by_rerank)/n:.1%})")
print("not in top-100:", len(not_in_top100), f"({len(not_in_top100)/n:.1%})")
print("sum:", len(cut_by_rerank) + len(not_in_top100))   # must equal 7,166

cut by rerank (was in top-100): 854 (11.9%)
not in top-100: 6312 (88.1%)
sum: 7166
